# Laboratorium nr 6 - Lokalne modele LLM z Ollama


## Pierwszy blok kodu - import bibliotek

In [5]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

## Drugi blok kodu - test czy działa ollama

In [8]:
message = "Tell me somethimg about transformer technology"
response = ollama.chat(model='llama3.2', messages=[{"role":"user", "content":message}])
print(response['message']['content'])

Transformer technology is a fascinating field that has revolutionized the way we transmit and distribute electricity. Here's an overview:

**What is a Transformer?**

A transformer is an electrical device that transfers electrical energy from one circuit to another through electromagnetic induction. It consists of two coils, known as primary and secondary coils, which are wound around a common core.

**How does it work?**

When an alternating current (AC) flows through the primary coil, it creates a magnetic field. This magnetic field induces a voltage in the secondary coil, which is connected to another circuit. The transformer's design allows for efficient transfer of energy between two circuits with different voltages or currents.

**Key Components:**

1. **Primary Coil:** The coil where the AC current flows.
2. **Secondary Coil:** The coil that receives the induced voltage.
3. **Core:** The common magnetic material used in the coils.
4. **Insulation:** Materials used to prevent ele

## Trzeci blok kodu - klasa Website (web scraper)

In [9]:
# Niektóre strony internetowe wymagają użycia odpowiednich nagłówków podczas ich pobierania:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# Klasa reprezentująca stronę internetową, która zostanie pobrana do podsumowania.
class Website:
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        # Użyjemy biblioteki BeautifulSoup do zbudowania obiektu z treścią
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input","link"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

## Czwarty blok kodu - sprawdzenie czy klasa działa

In [11]:
web = Website("https://pl.wikipedia.org/wiki/Duży_model_językowy")
print("Title: ", web.title)
print("text: ", web.text)

Title:  Duży model językowy – Wikipedia, wolna encyklopedia
text:  Przejdź do zawartości
Menu główne
Menu główne
przypnij
ukryj
Nawigacja
Strona główna
Losuj artykuł
Kategorie artykułów
Najlepsze artykuły
Częste pytania (FAQ)
Dla czytelników
O Wikipedii
Kontakt
Dla wikipedystów
Pierwsze kroki
Portal wikipedystów
Ogłoszenia
Zasady
Pomoc
Strony specjalne
Ostatnie zmiany
Szukaj
Szukaj
Wygląd
Wspomóż Wikipedię
Utwórz konto
Zaloguj się
Narzędzia osobiste
Wspomóż Wikipedię
Utwórz konto
Zaloguj się
Spis treści
przypnij
ukryj
Początek
1
Historia
2
Trenowanie i architektura
Przełącz podsekcję Trenowanie i architektura
2.1
Wzmacnianie z informacją zwrotną od człowieka
2.2
Mieszanka ekspertów
2.3
Dostrajanie na podstawie instrukcji
3
Koszt
4
Rozumowanie
5
Oddziaływanie
Przełącz podsekcję Oddziaływanie
5.1
Prawa autorskie
5.2
Bezpieczeństwo
5.3
Stronniczość algorytmiczna
5.4
Zdrowie psychiczne
6
Zobacz też
7
Przypisy
8
Linki zewnętrzne
Przełącz stan spisu treści
Duży model językowy
66 języków
Afri

## Piąty blok kodu - prompt systemowy

Zgodnie z zadaniem ostatnia linia każe modelowi odpowiadać w języku polskim (`Respond in markdown in Polish`).

In [12]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown in Polish."

## Szósty blok kodu - funkcja budująca prompt użytkownika

In [13]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

## Siódmy blok kodu - struktura wiadomości dla API modelu

In [14]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

Podgląd jak wygląda wiadomość przesyłana do modelu:

In [15]:
messages_for(web)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown in Polish.'},
 {'role': 'user',
  'content': "You are looking at a website titled Duży model językowy – Wikipedia, wolna encyklopedia\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\nPrzejdź do zawartości\nMenu główne\nMenu główne\nprzypnij\nukryj\nNawigacja\nStrona główna\nLosuj artykuł\nKategorie artykułów\nNajlepsze artykuły\nCzęste pytania (FAQ)\nDla czytelników\nO Wikipedii\nKontakt\nDla wikipedystów\nPierwsze kroki\nPortal wikipedystów\nOgłoszenia\nZasady\nPomoc\nStrony specjalne\nOstatnie zmiany\nSzukaj\nSzukaj\nWygląd\nWspomóż Wikipedię\nUtwórz konto\nZaloguj się\nNarzędzia osobiste\nWspomóż Wikipedię\nUtwórz konto\nZaloguj się\nSpis treści\nprzypnij\nukryj\nPoczątek\

## Ósmy blok kodu - funkcja podsumowywująca

In [36]:
MODEL='llama3.2'
def summarize(url):
    website = Website(url)
    response = ollama.chat(
        model=MODEL,
        messages=messages_for(website)
    )
    return response['message']['content']

Test funkcji:

In [37]:
summarize("https://wp.pl")

'Oto krótki podsumunek strony Wirtualna Polska - Wszystko co ważne:\n\n**Wiadomości**\n\n* Ostatnie doniesienia z dziedzin:\n\t+ Sport (mezcze, wyniki, wybory)\n\t+ Biznes i finanse (rynki, kursy walut, giełda)\n\t+ Polityka (wybory, aktoria, rządy)\n\t+ Rozrywka i styl życia (filmy, płyty, TV)\n* Wiadomości z dziedziny technologi:\n\t+ Nowe urządzenia komputerowe\n\t+ Programy i aplikacje\n\t+ Technologie w różnych dziedzinach\n\n**Strony poświęcone temu**\n\n* Sportowe Fakty\n* WP Finanse\n* WP Tech\n* Rozrywka i Styl życia\n* Telewizja Jastrząb\n* Radio Online\n* Wirtualny alkomat\n* Oglądaj ponad 120 kanałów TV przez internet\n\n**Ostatnie doniesienia**\n\n* Mundial 2026: wyniki, tabele, grupa I\n* Kolejne doniesienia z dziedziny sportu (mezcze, wybory)\n* Nowe produkty i usługi w dziedzinie technologi\n\n**Inne strony**\n\n* WP Radar Okazji\n* WP Kreator\n* PRACA.MONEY.PL\n* Dobreprogramy\n* Lucky Patcher'

## Dziewiąty blok kodu - wyświetlanie odpowiedzi w formacie markdown

In [38]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [39]:
display_summary("https://wp.pl")

**Podsumowanie witryny Wirtualna Polska**

Witryna Wirtualna Polska to polska platforma internetowa, która oferuje szereg usług i treści, w tym:

* Wiadomości i aktualizacje z dziedzin: sport, biznes, finanse, media, zdrowie, technologia, a także ogólnoświatowe wiadomości.
* Seria wydawnictw (np. WP SportoweFakty, WP Książki) oraz portalów internetowych (np. WP.pl)
* Serwis TV Online i Radio Online
* Sekcje o tematach takich jak turystyka, motoryzacja, sport, zdrowie, technologia itp.
* Blogi i strony dedykowane różnym tematom (np. kuchnia, moda, film)
* Reklamy i anonsy

**Wiadomości**

Witryna Wirtualna Polska oferuje regularne wiadomości z dziedzin: sportu, biznesu, finansów, mediali, zdrowia itp. Wiadomości są dostępne w formie artykułów i newsów.

**Sport**

Platforma oferuje szeroki zakres informacji o wydarzeniach sportowych, w tym wyniki meczów, notyfikacje o wyścigach, resultaty rozgrywek i informacje o zespołach i piłkarzach.

**Biznes i finanse**

Witryna oferuje newsy i informacje z dziedzin biznesu, finansów, gospodarki i inwestycji.

**Technologia**

Platforma oferuje informacje o nowych produktach i usługach, nagłoszenia o wyścigach i imprezach związanych z technologią, a także recenzje i review produktów.

## Zadanie - test różnych modeli i odpowiedzi w języku polskim

In [40]:
def summarize_with(url, model):
    website = Website(url)
    response = ollama.chat(
        model=model,
        messages=messages_for(website)
    )
    return response['message']['content']

for model in ['llama3.2', 'deepseek-r1:8b']:
    print(f"===== Model: {model} =====")
    display(Markdown(summarize_with("https://wp.pl", model)))

===== Model: llama3.2 =====


**Podsumowanie strony Wirtualna Polska - Wszystko co ważne**

Strona Wirtualna Polska - Wszystko co ważne to serwis internetowy, który oferuje wybrane treści z różnych kategori, takich jak:

* Wiadomości
	+ Newsy
	+ Wybory Prezydenckie
	+ Biznes i Finanse
	+ Sport
* Praca
	+ Oferty pracy
	+ Wynagrodzenia
	+ Technologie
* Rozrywka i Styl życia
	+ Kuchnia
	+ Film
	+ Horoskop
	+ Sporty Walki
* Inne
	+ Pogoda
	+ Motor
	+ Technologia
	+ Porady konsumenckie

Strona oferuje również wybrane rekordy i informacje o wydarzeniach, takich jak:

* Mundial 2026
* Wybory Prezydenckie
* Przepisy i rekomendacje z różnych kategorii

**Aktualne wiadomości**

Ponieważ strona ma wiele kategorii, nie jest możliwe przedstawienie wszystkich aktualnych wiadomości. Jednak można zebrac ich z kategori, w których są najbardziej aktywne.

* Wiadomości
	+ Newsy: "Mundial 2026", "Wybory Prezydenckie"
	+ Biznes i Finanse: "Rekordzista ma miliony długu", "Gigant szykuje się do wielkich cięć"
* Praca
	+ Oferty pracy: "100 tys. osób stracą pracę"
	+ Wynagrodzenia: "Wielki powrót Hołowni? Kulisy sensacyjnego scenariusza w Polsce 2050"
* Rozrywka i Styl życia
	+ Kuchnia: "Miliony wyświetleń, hit także w Polsce. Będzie kontynuacja"
	+ Sporty Walki: "Kolejna zmiana ws. Superpucharu. Jest komunikat PZPN"

===== Model: deepseek-r1:8b =====



# Podsumowanie strony www.wp.pl

Serwis internetowy **Wirtualna Polska** (wp.pl) jest ogólnopolskim portalem informacyjnym o szerokim spektrum tematów.

## Główne działy:
*   **Wiadomości:** Aktualności, polityka (w tym wybory prezydenckie), ekonomia, praca, wojsko, kultura, edukacja, medycyna.
*   **Sport:** Wyniki meczów (football, żużel itp.), transferы piłkarskie, mecze na żywo, materiały sportowe.
*   **Rozrywka i Styl:** Kwestionariusze, horoskopy, gazyngi, książki, filmy, telewizja, motoryzacja.
*   **Technologia:** Gadżety (telefony, komputery), programy, benchmarky sprzętu, wiadomości techniczne.

## Dodatkowe funkcje:
*   Relacje z zagranicy i wojska
*   Serwisy pogodowe dla wielu miast i regionów
*   Kalkulatory (np. LPG, wynagrodzeń)
*   Wiadomości turystyczne

Portal oferuje wiele rubryk informacyjnych z aktualnymi działaniami w nich następującymi oraz reklamy poszerzające jego ofertę tematyczną.